Cell 1 — Imports + FAISS check (for IVFPQ)

In [ ]:
import os
import time
import numpy as np
import pandas as pd

from typing import Dict, Tuple, List

import faiss
print("FAISS version:", faiss.__version__)

 FAISS version: 1.13.2


Cell 2 — Config (files + metrics + IVFPQ params + output)

In [2]:
# ---- Input embedding files ----
emb_folder = "\Embeddings"
EMBEDDING_FILES = {
    "bert_finetuned": emb_folder + "\\bert_finetuned_embeddings.xlsx",
    "gemini": emb_folder + "\\Gemini_Embedding.xlsx",
    "qwen3_8b": emb_folder + "\\Qwen3_Embedding_8B.xlsx",
    "sbert": emb_folder + "\\SBERT_Embedding_2_classification.xlsx",
}

TOPK_LIST = [1, 5, 10, 20]

# same similarity configs as before
SIM_METRICS = ["cosine", "dot", "L2"]

# ---- IVFPQ parameters ----
# coarse clusters
PQ_NLIST = 128

# number of sub-quantizers (M): must divide embedding dimension d
# 384 -> choose 48 (384/48=8 dims per subvector)
# 768 -> choose 96 (768/96=8 dims per subvector)
PQ_M_DEFAULT = None  # auto-select per embedding dimension

# bits per subvector code (nbits): 8 is standard (256 centroids per subquantizer)
PQ_NBITS = 8

# number of clusters to probe at query time
PQ_NPROBE = 10

OUT_XLSX = "faiss_ivfpq_results.xlsx"

Cell 3 — Loader

In [3]:
def is_numeric_col_name(c) -> bool:
    if isinstance(c, (int, np.integer)):
        return True
    s = str(c)
    return s.isdigit()

def load_embedding_xlsx(path: str) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    df = pd.read_excel(path, engine="openpyxl")

    # label column
    label_candidates = [c for c in df.columns if str(c).lower() in ("label", "y", "class")]
    if not label_candidates:
        raise ValueError(f"[{path}] No label column found.")
    label_col = label_candidates[0]

    # id column
    id_candidates = [c for c in df.columns if str(c).lower() in ("filename", "file", "text_file", "id", "file_id")]
    if id_candidates:
        preferred = [c for c in id_candidates if str(c).lower() in ("filename", "file", "text_file")]
        id_col = preferred[0] if preferred else id_candidates[0]
    else:
        non_num = [c for c in df.columns if c != label_col and not is_numeric_col_name(c)]
        if not non_num:
            raise ValueError(f"[{path}] No id-like column found.")
        id_col = non_num[0]

    # embedding columns
    emb_cols = [c for c in df.columns if c != label_col and is_numeric_col_name(c)]
    if not emb_cols:
        emb_cols = [c for c in df.columns
                    if c != label_col and str(c).lower().startswith("e") and str(c)[1:].isdigit()]

    if not emb_cols:
        raise ValueError(f"[{path}] No embedding columns found.")

    X = df[emb_cols].to_numpy(dtype=np.float32)
    y = df[label_col].to_numpy()
    ids = df[id_col].astype(str).to_numpy()

    return df, X, y, ids

Cell 4 — IR metrics

In [4]:
def precision_at_k(rels: np.ndarray, k: int) -> float:
    return float(np.sum(rels[:k])) / float(k)

def recall_at_k(rels: np.ndarray, k: int, total_relevant: int) -> float:
    if total_relevant <= 0:
        return 0.0
    return float(np.sum(rels[:k])) / float(total_relevant)

def dcg_at_k(rels: np.ndarray, k: int) -> float:
    rels_k = rels[:k]
    denom = np.log2(np.arange(2, k + 2))
    return float(np.sum(rels_k / denom))

def ndcg_at_k(rels: np.ndarray, k: int) -> float:
    dcg = dcg_at_k(rels, k)
    ideal = np.sort(rels)[::-1]
    idcg = dcg_at_k(ideal, k)
    return 0.0 if idcg == 0 else (dcg / idcg)

def mrr_at_k(rels: np.ndarray, k: int) -> float:
    rels_k = rels[:k]
    idx = np.where(rels_k == 1)[0]
    return 0.0 if len(idx) == 0 else (1.0 / float(idx[0] + 1))

Cell 5 — Helper: choose PQ_M (must divide dimension) + training helper

In [5]:
def choose_pq_m(d: int) -> int:
    """
    Choose M such that d % M == 0 and subvector dim (d/M) is reasonable.
    Prefer sub-dim 8, then 16, then 4.
    """
    preferred_subdims = [8, 16, 4]
    for sd in preferred_subdims:
        if d % sd == 0:
            return d // sd  # M = d / subdim
    # fallback: largest divisor <= 96
    for M in range(min(96, d), 1, -1):
        if d % M == 0:
            return M
    raise ValueError(f"Cannot find suitable PQ_M for d={d}")

def train_if_needed(index, X_train: np.ndarray):
    if not index.is_trained:
        index.train(X_train)

Cell 6 — IVFPQ Retrieval (L2 / dot / cosine) + PerQuery output

In [6]:
from sklearn.preprocessing import normalize

def faiss_ivfpq_eval_all3(
    X: np.ndarray, y: np.ndarray, ids: np.ndarray,
    topk_list: List[int],
    nlist: int,
    nprobe: int,
    nbits: int,
    M: int = None
) -> pd.DataFrame:
    """
    FAISS IVFPQ approximate search:
      - IVFPQ + L2          -> metric='L2'
      - IVFPQ + IP (dot)    -> metric='dot'
      - IVFPQ + IP on normalized vectors -> metric='cosine'
    Relevance: same label.
    """
    X = np.asarray(X, dtype=np.float32)
    N, d = X.shape

    if M is None:
        M = choose_pq_m(d)
    if d % M != 0:
        raise ValueError(f"PQ_M must divide d. Got d={d}, M={M}")

    maxK = max(topk_list)
    search_k = min(N, maxK + 1)  # +1 because self will appear

    rows = []

    def eval_one(metric_name: str, X_use: np.ndarray, faiss_metric: str):
        """
        faiss_metric: 'L2' or 'IP'
        """
        if faiss_metric == "L2":
            quantizer = faiss.IndexFlatL2(d)
            index = faiss.IndexIVFPQ(quantizer, d, nlist, M, nbits, faiss.METRIC_L2)
        elif faiss_metric == "IP":
            quantizer = faiss.IndexFlatIP(d)
            index = faiss.IndexIVFPQ(quantizer, d, nlist, M, nbits, faiss.METRIC_INNER_PRODUCT)
        else:
            raise ValueError("faiss_metric must be 'L2' or 'IP'")

        # train (required)
        train_if_needed(index, X_use)
        index.add(X_use)
        index.nprobe = nprobe

        # batch search
        D, I = index.search(X_use, search_k)

        for i in range(N):
            nbrs = I[i]
            nbrs = nbrs[nbrs != i]  # remove self
            if len(nbrs) == 0:
                continue
            nbrs = nbrs[:maxK]

            rel = (y[nbrs] == y[i]).astype(np.int32)
            total_rel = int(np.sum(y == y[i]) - 1)

            for K in topk_list:
                Ke = min(K, len(rel))
                rows.append({
                    "method": f"faiss_ivfpq_nlist{nlist}_m{M}_nbits{nbits}_nprobe{nprobe}",
                    "metric": metric_name,
                    "query_id": ids[i],
                    "query_label": int(y[i]),
                    "K": int(K),
                    "precision@K": precision_at_k(rel, Ke),
                    "recall@K": recall_at_k(rel, Ke, total_rel),
                    "ndcg@K": ndcg_at_k(rel, Ke),
                    "mrr@K": mrr_at_k(rel, Ke),
                })

    # 1) L2
    eval_one("L2", X, "L2")

    # 2) dot (inner product)
    eval_one("dot", X, "IP")

    # 3) cosine (inner product on normalized vectors)
    Xn = normalize(X, axis=1).astype(np.float32)
    eval_one("cosine", Xn, "IP")

    return pd.DataFrame(rows)

Cell 7 — Test IVFPQ on SBERT (Sanity check)

In [7]:
emb_name = "sbert"
path = EMBEDDING_FILES[emb_name]
df0, X0, y0, ids0 = load_embedding_xlsx(path)

M_sel = choose_pq_m(X0.shape[1])
print("Selected PQ_M:", M_sel, "| d:", X0.shape[1], "| sub-dim:", X0.shape[1] // M_sel)

t0 = time.time()
perq_sbert_pq = faiss_ivfpq_eval_all3(
    X0, y0, ids0,
    topk_list=TOPK_LIST,
    nlist=PQ_NLIST,
    nprobe=PQ_NPROBE,
    nbits=PQ_NBITS,
    M=None  # auto
)
print("Done IVFPQ:", emb_name, "| rows:", len(perq_sbert_pq), "| seconds:", round(time.time()-t0, 2))

print(perq_sbert_pq["metric"].value_counts())
perq_sbert_pq.head()

Selected PQ_M: 48 | d: 384 | sub-dim: 8
Done IVFPQ: sbert | rows: 27876 | seconds: 3.22
metric
L2        9292
dot       9292
cosine    9292
Name: count, dtype: int64


,method,metric,query_id,query_label,K,precision@K,recall@K,ndcg@K,mrr@K
0,faiss_ivfpq_nlist128_m48_nbits8_nprobe10,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,1,1.0,0.007463,1.0,1.0
1,faiss_ivfpq_nlist128_m48_nbits8_nprobe10,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,5,1.0,0.037313,1.0,1.0
2,faiss_ivfpq_nlist128_m48_nbits8_nprobe10,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,10,1.0,0.074627,1.0,1.0
3,faiss_ivfpq_nlist128_m48_nbits8_nprobe10,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,20,1.0,0.149254,1.0,1.0
4,faiss_ivfpq_nlist128_m48_nbits8_nprobe10,L2,1001_18_computer__wJS0wCqtEeiGfabwoJ7AXg.txt,18,1,1.0,0.006452,1.0,1.0


Cell 8 — Run ALL embeddings + Summary + Save Excel (IVFPQ)

In [ ]:
def summarize(perquery: pd.DataFrame, embedding_name: str) -> pd.DataFrame:
    grp = perquery.groupby(["method", "metric", "K"], as_index=False).agg({
        "precision@K": "mean",
        "recall@K": "mean",
        "ndcg@K": "mean",
        "mrr@K": "mean",
    })
    grp.insert(0, "embedding", embedding_name)
    return grp

all_perquery = []
all_summary = []

for emb_name, path in EMBEDDING_FILES.items():
    print(f"\n=== FAISS IVFPQ: {emb_name} ===")
    if not os.path.exists(path):
        print("File not found:", path)
        continue

    df_e, X, y, ids = load_embedding_xlsx(path)

    M_sel = choose_pq_m(X.shape[1])
    print(f"  Using M={M_sel} (sub-dim={X.shape[1]//M_sel})")

    t0 = time.time()
    perq = faiss_ivfpq_eval_all3(
        X, y, ids,
        topk_list=TOPK_LIST,
        nlist=PQ_NLIST,
        nprobe=PQ_NPROBE,
        nbits=PQ_NBITS,
        M=M_sel
    )
    perq.insert(0, "embedding", emb_name)
    all_perquery.append(perq)

    summ = summarize(perq, emb_name)
    all_summary.append(summ)

    print("  perquery rows:", len(perq), "| seconds:", round(time.time()-t0, 2))

df_perquery = pd.concat(all_perquery, ignore_index=True)
df_summary  = pd.concat(all_summary, ignore_index=True)

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as w:
    df_summary.to_excel(w, sheet_name="Summary", index=False)
    df_perquery.to_excel(w, sheet_name="PerQuery", index=False)

print("\nSaved:", OUT_XLSX)
df_summary.head(20)


=== FAISS IVFPQ: bert_finetuned ===
  Using M=96 (sub-dim=8)
   perquery rows: 27876 | seconds: 3.43

=== FAISS IVFPQ: gemini ===
  Using M=96 (sub-dim=8)
   perquery rows: 27876 | seconds: 6.9

=== FAISS IVFPQ: qwen3_8b ===
  Using M=512 (sub-dim=8)
   perquery rows: 27876 | seconds: 28.4

=== FAISS IVFPQ: sbert ===
  Using M=48 (sub-dim=8)
   perquery rows: 27876 | seconds: 3.26

 Saved: faiss_ivfpq_results.xlsx


,embedding,method,metric,K,precision@K,recall@K,ndcg@K,mrr@K
0,bert_finetuned,faiss_ivfpq_nlist128_m96_nbits8_nprobe10,L2,1,0.948773,0.013964,0.948773,0.948773
1,bert_finetuned,faiss_ivfpq_nlist128_m96_nbits8_nprobe10,L2,5,0.934051,0.068404,0.944328,0.959901
2,bert_finetuned,faiss_ivfpq_nlist128_m96_nbits8_nprobe10,L2,10,0.930263,0.135779,0.946988,0.960800
3,bert_finetuned,faiss_ivfpq_nlist128_m96_nbits8_nprobe10,L2,20,0.925312,0.269363,0.964191,0.961000
4,bert_finetuned,faiss_ivfpq_nlist128_m96_nbits8_nprobe10,cosine,1,0.950495,0.014010,0.950495,0.950495
5,bert_finetuned,faiss_ivfpq_nlist128_m96_nbits8_nprobe10,cosine,5,0.935084,0.068511,0.945100,0.960913
6,bert_finetuned,faiss_ivfpq_nlist128_m96_nbits8_nprobe10,cosine,10,0.929832,0.135792,0.946513,0.961420
7,bert_finetuned,faiss_ivfpq_nlist128_m96_nbits8_nprobe10,cosine,20,0.924365,0.269192,0.963804,0.961699
8,bert_finetuned,faiss_ivfpq_nlist128_m96_nbits8_nprobe10,dot,1,0.947912,0.013931,0.947912,0.947912
9,bert_finetuned,faiss_ivfpq_nlist128_m96_nbits8_nprobe10,dot,5,0.934912,0.068492,0.943456,0.957089
